In [2]:
import gc
import json
import math
import os
import random
import time
import warnings
import torch
import pywt
import importlib

import numpy as np
import pandas as pd
import torch.nn as nn

from pathlib import Path
# from google.colab import drive ### decomentat pentru collab
from dotenv import load_dotenv
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn import metrics as sklearn_metrics
from scipy import integrate
from scipy.signal import detrend as scipy_detrend

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# # Colab
# drive.mount("/content/drive", force_remount=True)

# # NAB
# nab = Path("/content/drive/MyDrive/NAB-master")
# nab_data = nab / "data"
# with open(nab / "labels" / "combined_windows.json") as f:
#     windows = json.load(f)

# # Yahoo
# yahoo_root = Path("/content/drive/MyDrive/Yahoo_S5_Data")

# Configuratie locala
load_dotenv()
nab = Path(os.getenv('NAB_PATH'))
nab_data = nab / 'data'
with open(nab / 'labels' / 'combined_windows.json') as f:
    windows = json.load(f)
yahoo_root = Path(os.getenv('YAHOO_PATH'))

# Configuratii NAB 
category_config = {
    "artificialWithAnomaly": {"interval": 600,  "smooth_frac": 0.003, "smoother": "db6"},
    "realAdExchange":        {"interval": 3600, "smooth_frac": 0.01,  "smoother": "sym4"},
    "realAWSCloudwatch":     {"interval": 600,  "smooth_frac": 0.01,  "smoother": "bior1.3"},
    "realTraffic":           {"interval": 600,  "smooth_frac": 0.003, "smoother": "coif3"},
    "realTweets":            {"interval": 600,  "smooth_frac": 0.003, "smoother": "db4"},
}

# Configuratii Yahoo S5
yahoo_config = {
    "A1Benchmark": {"interval": 3600, "smooth_frac": 0.01, "smoother": "ewma"},
    "A2Benchmark": {"interval": 3600, "smooth_frac": 0.01, "smoother": "ewma"},
    "A3Benchmark": {"interval": 3600, "smooth_frac": 0.01, "smoother": "ewma"},
    "A4Benchmark": {"interval": 3600, "smooth_frac": 0.01, "smoother": "ewma"},
}

WINDOW_SIZE = 100

Device: cuda


In [3]:
# Modelul AER (Auto-Encoder with Regression)
class AER(nn.Module):
    def __init__(self, window_size = 100, hidden_size = 30):
        super().__init__()
        self.window_size = window_size
        self.hidden_size = hidden_size
        self.latent_dim = hidden_size * 2

        self.encoder = nn.LSTM(
            input_size = 1,
            hidden_size = hidden_size,
            bidirectional = True,
            batch_first = True,
        )
        self.decoder = nn.LSTM(
            input_size = self.latent_dim,
            hidden_size = hidden_size,
            bidirectional = True,
            batch_first = True,
        )
        self.output_layer = nn.Linear(self.latent_dim, 1)

    def forward(self, x):
        _, (h, _) = self.encoder(x)
        latent = torch.cat([h[0], h[1]], dim=-1)

        repeated = latent.unsqueeze(1).repeat(1, self.window_size, 1)
        decoded, _ = self.decoder(repeated)

        output = self.output_layer(decoded).squeeze(-1)
        return output


In [4]:
def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def time_segments_aggregate(table, interval):
    start = table.index.values[0]
    maxi = table.index.values[-1]

    values, index = [], []
    while start <= maxi:
        end = start + interval
        aux = table.loc[start: end - 1]
        values.append(aux.mean().values)
        index.append(start)
        start = end

    return np.asarray(values), np.asarray(index)


def rolling_window(X, window_size, step_size = 1):
    new_x = []
    start = 0
    max_start = len(X) - window_size + 1
    while start < max_start:
        new_x.append(X[start: start + window_size])
        start += step_size
    return np.asarray(new_x)


def load_and_preprocess_nab(csv_path, interval = 600, window_size = 100):
    df = pd.read_csv(csv_path)
    df["timestamp"] = pd.to_datetime(df["timestamp"]).astype("int64") // 10**9
    df = df.sort_values("timestamp").set_index("timestamp")

    values, index = time_segments_aggregate(df, interval)
    values = SimpleImputer().fit_transform(values)
    values = MinMaxScaler(feature_range=(-1, 1)).fit_transform(values)
    X = rolling_window(values, window_size=window_size, step_size = 1)

    if len(X) < 10:
        return None, None
    return X, index


def load_and_preprocess_yahoo(csv_path, window_size = 100):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    ts_col = "timestamps" if "timestamps" in df.columns else "timestamp"

    exclude = {ts_col, "is_anomaly", "anomaly", "changepoint", "trend", "noise", "seasonality1", "seasonality2", "seasonality3"}
    val_col = next(c for c in df.columns if c not in exclude)

    df_work = df[[ts_col, val_col]].copy()
    df_work.columns = ["timestamp", "value"]
    df_work = df_work.sort_values("timestamp").set_index("timestamp")

    values = df_work[["value"]].values
    values = SimpleImputer().fit_transform(values)
    values = scipy_detrend(values, axis = 0)
    values = MinMaxScaler(feature_range = (-1, 1)).fit_transform(values)
    X = rolling_window(values, window_size=window_size, step_size = 1)
    index = df_work.index.values

    if len(X) < 10:
        return None, None
    return X, index


def load_yahoo_windows(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    ts_col = "timestamps" if "timestamps" in df.columns else "timestamp"
    lab_col = "is_anomaly" if "is_anomaly" in df.columns else "anomaly"

    df = df[[ts_col, lab_col]].copy()
    df.columns = ["timestamp", "label"]
    df = df.sort_values("timestamp").reset_index(drop = True)

    windows_list = []
    in_anomaly = False
    start_ts = None
    for _, row in df.iterrows():
        if row["label"] == 1 and not in_anomaly:
            in_anomaly = True
            start_ts = row["timestamp"]
        elif row["label"] == 0 and in_anomaly:
            in_anomaly = False
            windows_list.append((
                pd.Timestamp(int(start_ts), unit="s"),
                pd.Timestamp(int(row["timestamp"]), unit="s"),
            ))
    if in_anomaly:
        windows_list.append((
            pd.Timestamp(int(start_ts), unit="s"),
            pd.Timestamp(int(df["timestamp"].iloc[-1]), unit="s"),
        ))
    return windows_list


In [5]:
def aer_loss(output, r_true, y_true, f_true, reg_ratio = 0.5):
    # L = (gamma/2) * MSE(r) + (1 - gamma) * MSE(y) + (gamma/2) * MSE(f)
    r_pred = output[:, 0]
    y_pred = output[:, 1:-1]
    f_pred = output[:, -1]

    r_loss = torch.mean((r_pred - r_true) ** 2)
    y_loss = torch.mean((y_pred - y_true) ** 2)
    f_loss = torch.mean((f_pred - f_true) ** 2)

    return (reg_ratio / 2) * r_loss + (1 - reg_ratio) * y_loss + (reg_ratio / 2) * f_loss


def train_aer(window, epochs = 50, batch_size = 64, lr = 1e-3, val_split = 0.1, patience = 15, tol = 5e-3, verbose = True):
    window_tensor = torch.FloatTensor(window)
    train_data = window_tensor[:, 1:-1, :]
    r = window_tensor[:, 0, 0]
    y = window_tensor[:, 1:-1, 0]
    f = window_tensor[:, -1, 0]

    model = AER(window_size = WINDOW_SIZE).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr = lr)

    (x_train, x_eval,
    r_train, r_eval,
    y_train, y_eval,
    f_train, f_eval) = train_test_split(train_data, r, y, f, test_size = val_split, random_state = 42, shuffle = True)

    train_loader = DataLoader(
        TensorDataset(x_train, r_train, y_train, f_train),
        batch_size = batch_size, shuffle = True,
    )
    eval_loader = DataLoader(
        TensorDataset(x_eval, r_eval, y_eval, f_eval),
        batch_size = batch_size,
    )
    scheduler = ReduceLROnPlateau(optimizer, mode = "min", factor = 0.5, patience = 5, min_lr = 1e-5)

    n_eval = len(train_data) * val_split
    n_train = len(train_data) - n_eval
    best_val = float("inf")
    cnt = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x, r_t, y_t, f_t in train_loader:
            x = x.to(device)
            r_t, y_t, f_t = r_t.to(device), y_t.to(device), f_t.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = aer_loss(out, r_t, y_t, f_t)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * len(x)
        train_loss /= n_train

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, r_e, y_e, f_e in eval_loader:
                x = x.to(device)
                r_e, y_e, f_e = r_e.to(device), y_e.to(device), f_e.to(device)
                out = model(x)
                val_loss += aer_loss(out, r_e, y_e, f_e).item() * len(x)
        val_loss /= n_eval
        scheduler.step(val_loss)

        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            current_lr = optimizer.param_groups[0]["lr"]
            print(f"Epoch {epoch+1}/{epochs}, train= {train_loss:.6f}, val= {val_loss:.6f}, lr= {current_lr:.6f}")

        if val_loss < best_val - tol:
            best_val = val_loss
            cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            cnt += 1
            if cnt >= patience:
                if verbose:
                    print(f"Early stop at epoch {epoch+1}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.to(device)
    model.eval()
    return model


In [6]:
def predict_aer(model, X_windows):
    x = torch.FloatTensor(X_windows[:, 1:-1, :])
    loader = DataLoader(TensorDataset(x), batch_size = 256)

    all_ry, all_y, all_fy = [], [], []
    model.eval()
    with torch.no_grad():
        for (batch_x,) in loader:
            out = model(batch_x.to(device)).cpu().numpy()
            all_ry.append(out[:, 0:1])
            all_y.append(out[:, 1:-1, np.newaxis])
            all_fy.append(out[:, -1:])

    return np.concatenate(all_ry), np.concatenate(all_y), np.concatenate(all_fy)


def dtw_distance(x, y):
    n, m = len(x), len(y)
    dp = np.full((n + 1, m + 1), np.inf)
    dp[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = (x[i - 1] - y[j - 1]) ** 2
            dp[i, j] = cost + min(dp[i - 1, j], dp[i, j - 1], dp[i - 1, j - 1])
    return np.sqrt(dp[n, m])


def wavelet_denoise(signal, wavelet = "sym4", smoothing_fraction = 0.01):
    # Filtru wavelet trece-jos cu nivel adaptiv (zero-all details)
    
    signal = np.asarray(signal, dtype=float)
    n = len(signal)
    if n < 4:
        return signal

    span = max(1, int(smoothing_fraction * n))
    level = max(1, round(np.log2(max(2.0, float(span)))))

    max_level = pywt.dwt_max_level(n, wavelet)
    level = min(level, max_level)

    coeffs = pywt.wavedec(signal, wavelet, level=level)
    coeffs_smooth = [coeffs[0]] + [np.zeros_like(c) for c in coeffs[1:]]
    return pywt.waverec(coeffs_smooth, wavelet)[:n]


def smooth_signal(signal, smoothing_fraction, smoother):
    # alegem intre ewma si wavelet
    if smoother == "ewma":
        span = max(1, int(len(signal) * smoothing_fraction))
        return pd.Series(signal).ewm(span=span).mean().values
    else:
        out = wavelet_denoise(signal, wavelet = smoother, smoothing_fraction = smoothing_fraction)
        return np.clip(out, 0, None)


def regression_errors(y, y_pred, smoothing_window = 0.01, smoother = "ewma"):
    errors = np.abs(y - y_pred)[:, 0]
    errors = smooth_signal(errors, smoothing_window, smoother)
    return errors


def dtw_error(y, y_pred, score_window = 10):
    window_size = (score_window // 2) * 2 + 1
    pad = window_size // 2

    y_pad = np.pad(y, (pad, pad), "constant", constant_values=(0, 0))
    y_pred_pad = np.pad(y_pred, (pad, pad), "constant", constant_values=(0, 0))

    similarity = []
    i = 0
    while i < len(y) - window_size:
        true_data = y_pad[i: i + window_size].flatten()
        pred_data = y_pred_pad[i: i + window_size].flatten()
        similarity.append(dtw_distance(true_data, pred_data))
        i += 1

    errors = ([0] * pad + similarity + [0] * (len(y) - len(similarity) - pad))
    return errors


def reconstruction_errors(y, y_pred, smoothing_window = 0.01, score_window = 10, smoother = "ewma"):
    # pastram fractia originala pentru wavelet (smoothing_fraction)
    smoothing_fraction = smoothing_window
    sw_int = min(math.trunc(len(y) * smoothing_window), 200)

    true = [item[0] for item in y.reshape((y.shape[0], -1))]
    for item in y[-1][1:]:
        true.extend(item)

    predictions = []
    pred_length = y_pred.shape[1]
    num_errors = y_pred.shape[1] + (y_pred.shape[0] - 1)

    for i in range(num_errors):
        intermediate = []
        for j in range(max(0, i - num_errors + pred_length), min(i + 1, pred_length)):
            intermediate.append(y_pred[i - j, j])
        if intermediate:
            predictions.append(np.median(np.asarray(intermediate)))

    true = np.asarray(true)
    predictions = np.asarray(predictions)

    errors = dtw_error(true, predictions, score_window)

    if smoother == "ewma":
        # EWMA original Orion: rolling mean centrat
        errors = pd.Series(errors).rolling(sw_int, center=True, min_periods=sw_int // 2).mean().values
    else:
        errors = wavelet_denoise(np.asarray(errors, dtype=float), wavelet= smoother, smoothing_fraction =smoothing_fraction)
        errors = np.clip(errors, 0, None)
    return errors


def fixed_threshold(errors, k =4):
    return errors.mean() + k * errors.std()


def find_sequences(errors, epsilon, anomaly_padding):
    above = pd.Series(errors > epsilon)
    index_above = np.argwhere(above.values)
    for i in index_above.flatten():
        above[max(0, i - anomaly_padding): min(i + anomaly_padding + 1, len(above))] = True

    shift = above.shift(1).fillna(False)
    change = above != shift

    max_below = 0 if above.all() else max(errors[~above])

    index = above.index
    starts = index[above & change].tolist()
    ends = (index[~above & change] - 1).tolist()
    if len(ends) == len(starts) - 1:
        ends.append(len(above) - 1)

    return np.array([starts, ends]).T, max_below


def get_max_errors(errors, sequences, max_below):
    rows = [{"max_error": max_below, "start": -1, "stop": -1}]
    for start, stop in sequences:
        rows.append({
            "start": start, "stop": stop,
            "max_error": max(errors[start: stop + 1]),
        })
    return pd.DataFrame(rows).sort_values("max_error", ascending=False).reset_index(drop=True)


def prune_anomalies(max_errors, min_percent):
    next_error = max_errors["max_error"].shift(-1).iloc[:-1]
    max_error = max_errors["max_error"].iloc[:-1]

    increase = (max_error - next_error) / max_error
    too_small = increase < min_percent

    last_index = -1 if too_small.all() else max_error[~too_small].index[-1]
    return max_errors[["start", "stop", "max_error"]].iloc[0: last_index + 1].values


def compute_scores(pruned, errors, threshold, window_start):
    anomalies = []
    denominator = errors.mean() + errors.std()
    for row in pruned:
        score = (row[2] - threshold) / denominator
        anomalies.append([row[0] + window_start, row[1] + window_start, score])
    return anomalies


def merge_sequences(sequences):
    if len(sequences) == 0:
        return np.array([])

    sorted_seqs = sorted(sequences, key=lambda e: e[0])
    new_seqs = [sorted_seqs[0]]
    score = [sorted_seqs[0][2]]
    weights = [sorted_seqs[0][1] - sorted_seqs[0][0]]

    for seq in sorted_seqs[1:]:
        prev = new_seqs[-1]
        if seq[0] <= prev[1] + 1:
            score.append(seq[2])
            weights.append(seq[1] - seq[0])
            avg = np.average(score, weights=weights)
            new_seqs[-1] = (prev[0], max(prev[1], seq[1]), avg)
        else:
            score = [seq[2]]
            weights = [seq[1] - seq[0]]
            new_seqs.append(seq)

    return np.array(new_seqs)


def find_window_sequences(window, anomaly_padding, min_percent, window_start):
    threshold = fixed_threshold(window)
    seqs, max_below = find_sequences(window, threshold, anomaly_padding)
    max_errors = get_max_errors(window, seqs, max_below)
    pruned = prune_anomalies(max_errors, min_percent)
    return compute_scores(pruned, window, threshold, window_start)


def orion_find_anomalies(errors, index, window_size_portion = 0.33, window_step_size_portion = 0.1, min_percent = 0.1, anomaly_padding = 50):
    window_size = np.ceil(len(errors) * window_size_portion).astype("int")
    window_step_size = np.ceil(window_size * window_step_size_portion).astype("int")

    window_start = 0
    window_end = 0
    sequences = []

    while window_end < len(errors):
        window_end = window_start + window_size
        window = errors[window_start:window_end]
        sequences.extend(find_window_sequences(window, anomaly_padding, min_percent, window_start))
        window_start += window_step_size

    sequences = merge_sequences(sequences)

    anomalies = []
    for start, stop, score in sequences:
        anomalies.append([index[int(start)], index[int(stop)], score])
    return np.asarray(anomalies)


def bi_regression_errors(y, ry_pred, fy_pred, smoothing_window = 0.01, smoother = "ewma"):
    time_steps = len(y[0]) - 1
    mask_steps = int(smoothing_window * len(y))
    ry, fy = y[:, 0], y[:, -1]

    f_scores = regression_errors(fy, fy_pred, smoothing_window = smoothing_window, smoother = smoother)
    f_scores[:mask_steps] = 0
    f_scores = np.concatenate([np.zeros(time_steps), f_scores])

    r_scores = regression_errors(ry, ry_pred, smoothing_window = smoothing_window, smoother = smoother)
    r_scores[:mask_steps] = min(r_scores)
    r_scores = np.concatenate([r_scores, np.zeros(time_steps)])

    scores = f_scores + r_scores
    scores[time_steps + mask_steps:-time_steps] /= 2
    return scores


def point_wise_error(y, y_pred):
    return np.abs(y - y_pred)


def score_anomalies(y, ry_pred, y_pred, fy_pred, smoothing_window = 0.01, smoother = "ewma"):
    # Combinare MULT (paper Eq. 10): scoruri normalizate in [1, 2] si inmultite
    reg_scores = bi_regression_errors(y, ry_pred, fy_pred, smoothing_window= smoothing_window, smoother= smoother)
    try:
        rec_scores = reconstruction_errors(y[:, 1:-1], y_pred, smoothing_window= smoothing_window, smoother =smoother)
    except Exception:
        # fallback la eroare punctuala daca DTW esueaza
        rec_scores_raw = point_wise_error(
            np.asarray([item[0] for item in y[:, 1:-1].reshape((y.shape[0], -1))]),
            y_pred[:, :, 0].flatten()[:y.shape[0] * (y.shape[1] - 2)]
        )
        sw = min(int(len(rec_scores_raw) * smoothing_window), 200)
        rec_scores = pd.Series(rec_scores_raw).rolling(sw, center = True, min_periods =sw//2).mean().values

    mask_steps = int(smoothing_window * len(y))
    rec_scores[:mask_steps] = min(rec_scores)
    rec_scores = np.concatenate([np.zeros(1), rec_scores, np.zeros(1)])

    reg_n = MinMaxScaler((1, 2)).fit_transform(reg_scores.reshape(-1, 1)).flatten()
    rec_n = MinMaxScaler((1, 2)).fit_transform(rec_scores.reshape(-1, 1)).flatten()
    return reg_n * rec_n


In [7]:
def overlap(expected, observed):
    return expected[0] < observed[1] and expected[1] > observed[0]


def overlap_segment(expected, observed):
    tp, fp, fn = 0, 0, 0
    observed_copy = observed.copy()
    for expected_seq in expected:
        found = False
        for observed_seq in observed:
            if overlap(expected_seq, observed_seq):
                if not found:
                    tp += 1
                    found = True
                if observed_seq in observed_copy:
                    observed_copy.remove(observed_seq)
        if not found:
            fn += 1
    fp += len(observed_copy)
    return fp, fn, tp


def to_intervals(df_or_list):
    if isinstance(df_or_list, list):
        return [(p[0], p[1] + 1) for p in df_or_list]
    return [(p[0], p[1] + 1) for p in df_or_list[["start", "end"]].itertuples(index=False)]


def contextual_metrics(expected_df, observed_df):
    expected = to_intervals(expected_df)
    observed = to_intervals(observed_df)
    fp, fn, tp = overlap_segment(expected, observed)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1, tp, fp, fn


def to_windows_df(wins):
    if not wins:
        return pd.DataFrame(columns=["start", "end"])
    return pd.DataFrame([{
        "start": int(pd.Timestamp(w[0]).timestamp()),
        "end":   int(pd.Timestamp(w[1]).timestamp()),
    } for w in wins])


def evaluate_detection(anomalies, expected_df):
    if len(anomalies) > 0:
        detected = pd.DataFrame(anomalies[:, :2], columns=["start", "end"])
    else:
        detected = pd.DataFrame(columns=["start", "end"])

    pre, rec, f1, tp, fp, fn = contextual_metrics(expected_df, detected)
    return {
        "precision": round(pre, 4),
        "recall": round(rec, 4),
        "f1": round(f1, 4),
        "tp": tp, "fp": fp, "fn": fn,
    }


def score_and_detect(X_windows, ry_pred, y_pred, fy_pred, index, sm_w, smoother = "ewma"):
    scores = score_anomalies(X_windows, ry_pred, y_pred, fy_pred, smoothing_window = sm_w, smoother = smoother)
    anomalies = orion_find_anomalies( errors = scores, index = index, window_size_portion =0.33, window_step_size_portion = 0.1,
        min_percent = 0.1, anomaly_padding = 50,
    )
    return anomalies


def run_pipeline_nab(csv_path, category, seed, verbose = False):
    cfg = category_config[category]
    key = f"{category}/{Path(csv_path).name}"

    set_seeds(seed)
    result = load_and_preprocess_nab(csv_path, interval= cfg["interval"], window_size =WINDOW_SIZE)
    if result[0] is None:
        return {"category": category, "file": Path(csv_path).name,
                "precision": 0.0, "recall": 0.0, "f1": 0.0,
                "tp": 0, "fp": 0, "fn": 0}

    X_windows, index = result
    model = train_aer(X_windows, epochs = 50, batch_size= 64, val_split = 0.1, verbose = verbose)
    ry_pred, y_pred, fy_pred = predict_aer(model, X_windows)

    anomalies = score_and_detect(X_windows, ry_pred, y_pred, fy_pred, index, sm_w =cfg["smooth_frac"], smoother =cfg.get("smoother", "ewma"))
    expected_df = to_windows_df(windows.get(key, []))
    eval_result = evaluate_detection(anomalies, expected_df)

    del model, X_windows
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    eval_result["category"] = category
    eval_result["file"] = Path(csv_path).name
    return eval_result


def eval_nab(seed = 0, verbose = False):
    rows = []
    for cat in category_config:
        cat_dir = nab_data / cat
        if not cat_dir.exists():
            continue
        if verbose:
            print(f"\n {cat}:")
        for csv_file in sorted(cat_dir.glob("*.csv")):
            t0 = time.time()
            result = run_pipeline_nab(csv_file, cat, seed, verbose = verbose)
            rows.append(result)
            print(f"{csv_file.name:<50} F1={result['f1']:.4f} ({time.time()-t0:.0f}s)")
    return pd.DataFrame(rows)


def micro_f1(df):
    g = df.groupby("category")[["tp", "fp", "fn"]].sum()
    g["precision"] = g["tp"] / (g["tp"] + g["fp"]).replace(0, 1)
    g["recall"]    = g["tp"] / (g["tp"] + g["fn"]).replace(0, 1)
    g["f1"]        = 2 * g["precision"] * g["recall"] / (g["precision"] + g["recall"]).replace(0, 1)
    return g[["precision", "recall", "f1"]].fillna(0).round(4)


In [8]:
def run_pipeline_yahoo(csv_path, category, seed, verbose = False):
    cfg = yahoo_config[category]

    set_seeds(seed)
    result = load_and_preprocess_yahoo(csv_path, window_size = WINDOW_SIZE)
    if result[0] is None:
        return {"category": category, "file": Path(csv_path).name,
                "precision": 0.0, "recall": 0.0, "f1": 0.0,
                "tp": 0, "fp": 0, "fn": 0}

    X_windows, index = result
    model = train_aer(X_windows, epochs = 50, batch_size = 64, val_split = 0.1, verbose = verbose)
    ry_pred, y_pred, fy_pred = predict_aer(model, X_windows)

    anomalies = score_and_detect(X_windows, ry_pred, y_pred, fy_pred, index, sm_w = cfg["smooth_frac"], smoother = cfg.get("smoother", "ewma"))
    expected_df = to_windows_df(load_yahoo_windows(csv_path))
    eval_result = evaluate_detection(anomalies, expected_df)

    del model, X_windows
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    eval_result["category"] = category
    eval_result["file"] = Path(csv_path).name
    return eval_result


def eval_yahoo(seed = 0, verbose = False):
    rows = []
    for cat in yahoo_config:
        cat_dir = yahoo_root / cat
        if not cat_dir.exists():
            print(f"Lipseste: {cat_dir}")
            continue
        if verbose:
            print(f"\n {cat}:")
        for csv_file in sorted(cat_dir.glob("*.csv")):
            if "_all" in csv_file.name:
                continue
            t0 = time.time()
            result = run_pipeline_yahoo(csv_file, cat, seed, verbose = verbose)
            rows.append(result)
            print(f"{csv_file.name:<50} F1={result['f1']:.4f} ({time.time()-t0:.0f}s)")
    return pd.DataFrame(rows)


In [9]:
# ===== NAB: 25 seeds =====
seeds = list(range(25))
paper_nab = {
    "artificialWithAnomaly": 0.615, "realAdExchange": 0.635,
    "realAWSCloudwatch": 0.621, "realTraffic": 0.606, "realTweets": 0.585,
}

all_seed_summaries = []

for seed in seeds:
    print(f"\n=== NAB Seed: {seed}/{len(seeds)-1} ===")
    df = eval_nab(seed)
    df.to_csv(f"nab_seed_{seed}.csv", index=False)

    micro = micro_f1(df)
    row = {"seed": seed}
    for cat in paper_nab:
        row[cat] = micro.loc[cat, "f1"] if cat in micro.index else 0.0
    row["overall"] = round(pd.Series([row[cat] for cat in paper_nab]).mean(), 4)
    all_seed_summaries.append(row)

    print(f"\nSeed {seed}: ", end="")
    for cat in paper_nab:
        print(f"{cat[:10]}={row[cat]:.4f}  ", end="")
    print(f"overall={row['overall']:.4f}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_seeds = pd.DataFrame(all_seed_summaries)
df_seeds.to_csv("nab_all_seeds.csv", index=False)

print(f"\nNAB results ({len(seeds)} seeds)")
print(df_seeds.to_string(index=False))

best_idx = df_seeds["overall"].idxmax()
print(f"\nBest seed: {int(df_seeds.loc[best_idx, 'seed'])}  (overall F1={df_seeds.loc[best_idx, 'overall']:.4f})")

print(f"\n{'Categorie':<28} {'Paper':>7} {'Best':>7} {'Mean':>7} {'Std':>7}")
for cat in paper_nab:
    print(f"{cat:<28} {paper_nab[cat]:>7.3f} {df_seeds[cat].max():>7.3f} {df_seeds[cat].mean():>7.3f} {df_seeds[cat].std():>7.3f}")
print(f"{'overall':<28} {'-':>7} {df_seeds['overall'].max():>7.3f} {df_seeds['overall'].mean():>7.3f} {df_seeds['overall'].std():>7.3f}")



=== NAB Seed: 0/24 ===
art_daily_flatmiddle.csv                           F1=0.5000 (7s)
art_daily_jumpsdown.csv                            F1=0.0000 (6s)
art_daily_jumpsup.csv                              F1=1.0000 (6s)
art_daily_nojump.csv                               F1=0.6667 (6s)
art_increase_spike_density.csv                     F1=1.0000 (5s)
art_load_balancer_spikes.csv                       F1=0.5000 (3s)
exchange-2_cpc_results.csv                         F1=0.0000 (3s)
exchange-2_cpm_results.csv                         F1=0.8000 (3s)
exchange-3_cpc_results.csv                         F1=1.0000 (2s)
exchange-3_cpm_results.csv                         F1=0.5000 (2s)
exchange-4_cpm_results.csv                         F1=0.7500 (2s)
ec2_cpu_utilization_24ae8d.csv                     F1=0.6667 (3s)
ec2_cpu_utilization_53ea38.csv                     F1=1.0000 (3s)
ec2_cpu_utilization_5f5533.csv                     F1=1.0000 (3s)
ec2_cpu_utilization_77c1ca.csv                     F

KeyboardInterrupt: 

In [ ]:
# ===== Yahoo S5: 10 seeds =====
seeds = list(range(10))
yahoo_cats = list(yahoo_config.keys())

all_yahoo_summaries = []

for seed in seeds:
    print(f"\n=== Yahoo Seed: {seed}/{len(seeds)-1} ===")
    df = eval_yahoo(seed)
    df.to_csv(f"yahoo_seed_{seed}.csv", index=False)

    micro = micro_f1(df)
    row = {"seed": seed}
    for cat in yahoo_cats:
        row[cat] = micro.loc[cat, "f1"] if cat in micro.index else 0.0
    row["overall"] = round(pd.Series([row[cat] for cat in yahoo_cats]).mean(), 4)
    all_yahoo_summaries.append(row)

    print(f"\nSeed {seed}: ", end="")
    for cat in yahoo_cats:
        print(f"{cat}={row[cat]:.4f}  ", end="")
    print(f"overall={row['overall']:.4f}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_yahoo = pd.DataFrame(all_yahoo_summaries)
df_yahoo.to_csv("yahoo_all_seeds.csv", index=False)

print(f"\nYahoo results ({len(seeds)} seeds)")
print(df_yahoo.to_string(index=False))

best_idx = df_yahoo["overall"].idxmax()
print(f"\nBest seed: {int(df_yahoo.loc[best_idx, 'seed'])}  (overall F1={df_yahoo.loc[best_idx, 'overall']:.4f})")

print(f"\n{'Categorie':<15} {'Best':>7} {'Mean':>7} {'Std':>7}")
for cat in yahoo_cats:
    print(f"{cat:<15} {df_yahoo[cat].max():>7.3f} {df_yahoo[cat].mean():>7.3f} {df_yahoo[cat].std():>7.3f}")
print(f"{'overall':<15} {df_yahoo['overall'].max():>7.3f} {df_yahoo['overall'].mean():>7.3f} {df_yahoo['overall'].std():>7.3f}")
